# Phase 1.3 — Signal Processing

I study time and frequency descriptions of EEG signals and compare raw and filtered oscillations.

> **Research integrity:** Figures labeled synthetic are not clinical EEG observations. Run code locally in VS Code using the project `.venv` interpreter.

## 1. Imports and synthetic signal

A stationary mixture is useful for verifying the mathematics, but does not represent a complete sleep stage.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from eeg_sleep.processing import bandpass, power_spectrum, time_frequency
fs = 100.0
t = np.arange(0, 30, 1/fs)
rng = np.random.default_rng(42)
x = 18*np.sin(2*np.pi*2*t) + 8*np.sin(2*np.pi*10*t) + 3*rng.normal(size=len(t))
print(f"Samples: {len(x)}, interval: {1/fs:.3f} s")

## 2. Continuous-time idea and discrete samples

With sampling frequency `fs`, discrete sample `n` corresponds to time `n / fs`. Compare a short waveform segment.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t[:200], x[:200], label="Synthetic mixture")
ax.set(xlabel="Time (s)",ylabel="Amplitude (µV)",title="Two seconds of synthetic EEG-like activity")
ax.legend(); plt.show()

## 3. Butterworth filtering

Zero-phase filtering is convenient for offline analysis. Filtering an isolated epoch creates boundary transients; the real EDF pipeline filters continuous recordings first.

In [ ]:
filtered = bandpass(x, fs=fs, low=0.5, high=40)
fig, ax = plt.subplots(2,1,sharex=True,figsize=(10,4))
ax[0].plot(t[:500],x[:500],linewidth=.8);ax[0].set_title("Before filtering")
ax[1].plot(t[:500],filtered[:500],linewidth=.8);ax[1].set_title("0.5–40 Hz filtered")
ax[1].set_xlabel("Time (s)");plt.tight_layout();plt.show()

## 4. Power spectral density using Welch

Welch PSD is an estimate of power per unit frequency. A high spectral peak is **not** enough to label an entire epoch as a sleep stage.

In [ ]:
f, psd = power_spectrum(filtered,fs=fs,nperseg=512)
fig,ax=plt.subplots(figsize=(9,3))
ax.plot(f,10*np.log10(np.maximum(psd,1e-12)))
ax.set(xlim=(0,45),xlabel="Frequency (Hz)",ylabel="Power (dB µV²/Hz)",title="Welch PSD — simulated signal")
plt.show()
print("Dominant estimated frequency:", f[np.argmax(psd)], "Hz")

## 5. Spectrogram: how spectral content changes over time

The time-frequency view becomes especially important for non-stationary patterns such as sleep spindles.

In [ ]:
frequencies, times, power = time_frequency(filtered,fs=fs,nperseg=256)
fig,ax=plt.subplots(figsize=(9,3))
mesh=ax.pcolormesh(times,frequencies,10*np.log10(np.maximum(power,1e-12)),shading="auto")
ax.set(ylim=(0,40),xlabel="Time (s)",ylabel="Frequency (Hz)",title="Spectrogram — simulated mixture")
fig.colorbar(mesh,ax=ax,label="dB µV²/Hz");plt.show()

## 6. Aliasing demonstration

At 100 Hz sampling, a 70 Hz sinusoid produces samples indistinguishable from an appropriate lower-frequency sinusoid. Anti-alias filtering is therefore necessary *before* acquisition.

In [ ]:
fs=100
n=np.arange(200)
high=np.sin(2*np.pi*70*n/fs)
low=np.sin(2*np.pi*30*n/fs)
print("Max |70Hz + 30Hz|:", np.max(np.abs(high+low)))
fig,ax=plt.subplots(figsize=(9,3))
ax.plot(n[:30]/fs,high[:30],"o-",label="70 Hz sampled")
ax.plot(n[:30]/fs,-low[:30],"x--",label="-30 Hz sampled")
ax.set(xlabel="Time (s)",ylabel="Amplitude",title="Aliasing at 100 Hz")
ax.legend();plt.show()

## Exercises

1. Write the sampling formula and Nyquist frequency for 100 Hz. 2. Repeat with 5 Hz and 35 Hz components. 3. Explain the difference between filtering in a real recording and filtering each epoch in isolation. 4. What are the limitations of Welch PSD for brief transient events?